In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder , LabelEncoder , OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import missingno
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline


# Task 1 : Exploratory Data Analysis (EDA)

In [ ]:
data = pd.read_csv(r'Fifa.csv')
data

In [ ]:
df = data.copy()

* Are there any missing values? How many, and in which columns?

In [ ]:
missingno.matrix(df , figsize=(12,5))
plt.show()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    print(f'nunique {col} : {df[col].nunique() }\n')

    top = df[col].value_counts().sort_values(ascending=False).head()
    down = df[col].value_counts().sort_values(ascending=False).tail()

    print(f'top : \n {top}')
    print(f'\n tail : \n {down}')
    print('\n')


### `age`:
*    41    8
*    42    4
*    44    1
*    43    1
*    15    1

### `future`:
*    92    4
*    47    2
*    94    1
*    95    1
*    46    1

`Value Per M$`
*    77.500    1
*    61.000    1
*    79.500    1
*    0.009     1
*    0.005     1

`total_stat`
*    758     1
*    805     1
*    792     1
*    1086    1
*    1087    1

In [ ]:
df[df['Age'].isin([41, 42, 44, 43, 15])]

In [ ]:
df[df['Overall_Rating'].isin([45, 44, 90, 36 ,38])]

In [ ]:
df[df['Future Potential'].isin([45, 44, 90, 36 ,38])]

In [ ]:
df[df['Value Per M$'].isin([77.500, 61.000, 79.500, 0.009, 0.0051])]

In [ ]:
df[df['Total_Stats Score'].isin([758, 805, 792, 1086, 1087])]

In [ ]:
mask_1 = df['Country'] == 'Egypt'
df[((mask_1) & df['Name'].str.contains(r'Mohamed' , case=False))]

In [ ]:
df.select_dtypes('object').describe()

In [ ]:
df.select_dtypes('number').describe()

* What does the distribution of Value Per M$ look like? Is it approximately normal or skewed? Compute the skewness.

In [ ]:
df.select_dtypes('number').skew()

In [ ]:
df.select_dtypes('number').skew().iloc[3]

In [ ]:
fig, ax = plt.subplots(5, 4, figsize=(18, 15))
ax = ax.flatten()

i = 0

for col in data.select_dtypes('number').columns:
    # الأصل
    sns.histplot(data[col], kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1

    # log transform (مع معالجة القيم =0)
    sns.histplot(np.log1p(data[col]), kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1

    sns.boxplot(data[col], ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1

    # log transform (مع معالجة القيم =0)
    sns.boxplot(np.log1p(data[col]), ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1

# نحذف أي subplots فاضية
for j in range(i, len(ax)):
    fig.delaxes(ax[j])

plt.tight_layout()
plt.show()

In [ ]:
data[data['Value Per M$'] == 0]

In [ ]:
data[data['Position'].isin(['SW','RF'])]

In [ ]:
df['Value Per M$'] =df['Value Per M$'].drop(index=df[df['Value Per M$'] == 0].index)
df = df.dropna(subset='Value Per M$')

In [ ]:
fig, ax = plt.subplots(5, 4, figsize=(18, 15))
ax = ax.flatten()

i = 0

for col in df.select_dtypes('number').columns:
    
    sns.histplot(df[col], kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1

    sns.histplot(np.log1p(df[col]), kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1

    sns.boxplot(df[col], ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1


    sns.boxplot(np.log1p(df[col]), ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1


for j in range(i, len(ax)):
    fig.delaxes(ax[j])

plt.tight_layout()
plt.show()

* Which numerical features seem most related to Value Per M$?

In [ ]:
corr = df.select_dtypes('number').corr()
corr

In [ ]:
corr['Value Per M$'].sort_values(ascending=False).to_frame().iloc[1:3 , 0]

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(corr , annot=True , fmt='.1f' , cmap='Blues')

* What is the average Overall_Rating per Position?

In [ ]:
fig_1 = data.groupby('Position')['Overall_Rating'].mean().to_frame().sort_values(by='Overall_Rating' , ascending=False)
fig_1_handle = df.groupby('Position')['Overall_Rating'].mean().to_frame().sort_values(by='Overall_Rating' , ascending=False)
fig_1

In [ ]:
fig_1_handle

In [ ]:
fig_1 = data.groupby('Position')['Overall_Rating'].mean().to_frame().sort_values(by='Overall_Rating', ascending=False)
fig_1_handle = df.groupby('Position')['Overall_Rating'].mean().to_frame().sort_values(by='Overall_Rating', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))


fig_1.plot(kind='bar', color='g', ax=axes[0])
axes[0].set_title('before handle market value')


fig_1_handle.plot(kind='bar', color='b', ax=axes[1])
axes[1].set_title('after handle market value')

plt.tight_layout()
plt.show()

In [ ]:
df_analysis = df.copy()

# Task 2 : Data Preprocessing

In [ ]:
stats.boxcox(df['Value Per M$'])

In [ ]:
df['Value Per M$_after_log'] = np.log1p(df['Value Per M$'])
df['Value Per M$_after_boxcox'], lam = stats.boxcox(df['Value Per M$'])

In [ ]:
corr = df.select_dtypes('number').corr()
corr

In [ ]:
corr[['Value Per M$', 'Value Per M$_after_log', 'Value Per M$_after_boxcox']]

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(corr[['Value Per M$', 'Value Per M$_after_log', 'Value Per M$_after_boxcox']] , annot=True , fmt='.1f' , cmap='Blues')

In [ ]:
corr[['Value Per M$_after_boxcox']].sort_values(by='Value Per M$_after_boxcox' , ascending=False).iloc[2:4 , :]

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(corr , annot=True , fmt='.1f' , cmap='Blues')

In [ ]:
fig,ax = plt.subplots(1,2 , figsize = (10,6))
sns.histplot(df['Value Per M$_after_log'] , ax=ax[0])
sns.histplot(df['Value Per M$_after_boxcox'] , ax=ax[1])

In [ ]:
df.select_dtypes('number').skew()

In [ ]:
def detect_outlairs(df):
    bounds = {}
    for col in df.select_dtypes('number').columns:
        q1 = df[col].quantile( .25)
        q3 = df[col].quantile( .75)
        iqr = q3 - q1

        lower_bound = q1 - (iqr*1.5)
        upper_bound = q3 + (iqr*1.5)

        bounds[col]  = (lower_bound , upper_bound)
    return bounds


def view_outlairs(df , bounds):
    for col in df.select_dtypes('number').columns:
        print(f'{col} : \n')

        lower_bound , upper_bound = bounds[col]
        print(f'upper : {upper_bound}')
        print(f'lower : {lower_bound}')
        
        outlairs_df = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col].copy()
        
        print(f'{col} outlairs rows {outlairs_df.shape[0]} \n')
        print(f'{outlairs_df} \n')
        print('='*100)
        print('\n')

In [ ]:
bounds = detect_outlairs(df)
view_outlairs(df , bounds)

In [ ]:
features_label = df.select_dtypes('object').drop(columns = ['Name'])
features_label

In [ ]:
df_pre= df.copy()

In [ ]:
labels = {}

for col in df.select_dtypes('object').drop(columns = 'Name').columns:
    label = LabelEncoder()
    df_pre[col+' label'] = label.fit_transform(df[col])

    labels[col] = label


In [ ]:
scalers = {}

for col in df.select_dtypes('number').columns:
    scaler = StandardScaler()
    df_pre[col + ' scaled'] = scaler.fit_transform(df[[col]])

    scalers[col] = scaler


In [ ]:
df = df_pre.copy()

In [ ]:
df.columns

In [ ]:
data.shape

In [ ]:
df = df.drop(columns = ['Name' , 'Country' , 'Position' , 'Age' ,
                    'Overall_Rating' , 'Future Potential', 'Team',
                    'Value Per M$', 'Total_Stats Score','Value Per M$_after_log', 'Value Per M$_after_boxcox',
                    'Value Per M$ scaled', 'Value Per M$_after_log scaled'])

In [ ]:
df

# Task 3 : Create Classification Target (Logistic Regression + Naïve Bayes)

In [ ]:
df['Performance_Tier'] = pd.qcut(
    df['Overall_Rating scaled'],
    q=4,
    labels=['Low', 'Average', 'Good', 'Elite']
)
df.head()

In [ ]:
class_counts = df['Performance_Tier'].value_counts().sort_values()
print(class_counts)

In [ ]:
class_counts.plot(kind='bar')
plt.title('Distribution of Performance Tiers')
plt.xlabel('Tier')
plt.ylabel('Number of Players')
plt.xticks(rotation=0)
plt.show()

In [ ]:
map_Performance_Tier = {'Low' : 0, 'Average' : 1,
                        'Good' : 2, 'Elite' : 3}
df['Performance_Tier_encoded'] = (df['Performance_Tier']).map(map_Performance_Tier)

In [ ]:
df.head()

In [ ]:
y = df['Value Per M$_after_boxcox scaled']
x = df.drop(columns=['Performance_Tier', 'Performance_Tier_encoded', 'Value Per M$_after_boxcox scaled'])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
linear = LinearRegression()
linear.fit(x_train, y_train)

y_pred_train = linear.predict(x_train)
y_pred_test = linear.predict(x_test)

print("Train R2:", r2_score(y_train, y_pred_train))
print("Train MAE:", mean_absolute_error(y_train, y_pred_train))
print("Train MSE:", mean_squared_error(y_train, y_pred_train))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)) , '\n')

print("Test R2:", r2_score(y_test, y_pred_test))
print("Test MAE:", mean_absolute_error(y_test, y_pred_test))
print("Test MSE:", mean_squared_error(y_test, y_pred_test))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

In [ ]:
poly = PolynomialFeatures(degree=2)

In [ ]:
pd.DataFrame(poly.fit_transform(x) , columns = poly.get_feature_names_out())

In [ ]:
degrees = [1, 2, 3, 4]

train_r2 = []
test_r2 = []

for d in degrees:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('lr', LinearRegression())
    ])
    
    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    train_r2.append(r2_score(y_train, y_train_pred))
    test_r2.append(r2_score(y_test, y_test_pred))

    print(f"\nDegree {d}")
    print("Train R2:", r2_score(y_train, y_train_pred))
    print("Test R2:", r2_score(y_test, y_test_pred))

In [ ]:
plt.plot(degrees, train_r2, label='Train R2')
plt.plot(degrees, test_r2, label='Test R2')

plt.xlabel("Polynomial Degree")
plt.ylabel("R2 Score")
plt.title("Polynomial Degree vs Performance")
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import Ridge

alphas = np.logspace(-4, 4, 20)

ridge_train_rmse = []
ridge_test_rmse = []

for a in alphas:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)), 
        ('ridge', Ridge(alpha=a))
    ])

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    ridge_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
    ridge_test_rmse.append(root_mean_squared_error(y_test, y_test_pred))

    
    print("Train R2:", r2_score(y_train, y_train_pred))
    print("Test R2:", r2_score(y_test, y_test_pred))

In [ ]:
plt.plot(alphas, ridge_test_rmse, label='Test RMSE')
plt.xscale('log')

plt.xlabel("Alpha")
plt.ylabel("RMSE")
plt.title("Ridge Regularization")
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import Lasso

lasso_train_rmse = []
lasso_test_rmse = []

for a in alphas:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('lasso', Lasso(alpha=a, max_iter=5000))
    ])

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    lasso_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
    lasso_test_rmse.append(root_mean_squared_error(y_test, y_test_pred))

    print(f"\nDegree {d}")
    print("Train R2:", r2_score(y_train, y_train_pred))
    print("Test R2:", r2_score(y_test, y_test_pred))

In [ ]:
plt.plot(alphas, lasso_test_rmse, label='Test RMSE')
plt.xscale('log')

plt.xlabel("Alpha")
plt.ylabel("RMSE")
plt.title("Lasso Regularization")
plt.legend()
plt.show()

In [ ]:
lasso_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lasso', Lasso(alpha=0.01))
])

lasso_model.fit(x_train, y_train)

coef = lasso_model.named_steps['lasso'].coef_

zero_features = np.sum(coef == 0)

print("Zero coefficients:", zero_features)

In [ ]:
coef

In [ ]:
x = df.drop(['Overall_Rating scaled', 'Performance_Tier', 'Performance_Tier_encoded'], axis=1)
y = df['Performance_Tier_encoded']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
x

In [ ]:
lr = LogisticRegression(max_iter=5000)
lr.fit(x_train, y_train)

y_pred = lr.predict(x_test)

LR_cm = confusion_matrix(y_test, y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(LR_cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print("Train accuracy:", lr.score(x_train, y_train))
print("Test accuracy:", lr.score(x_test, y_test))


In [ ]:
C_values = np.logspace(-3, 3, 10)

train_acc = {}
test_acc = {}

for c in C_values:
    model = LogisticRegression(C=c, max_iter=1000)
    model.fit(x_train, y_train)

    train_acc[c] = (model.score(x_train, y_train))
    test_acc[c] = (model.score(x_test, y_test))

    

In [ ]:
train_acc_list = [i for i in train_acc.values()]

In [ ]:
test_acc_list = [i for i in test_acc.values()]

In [ ]:
plt.plot(C_values, train_acc_list, label='Train Accuracy')
plt.plot(C_values, test_acc_list, label='Test Accuracy')

plt.xscale('log')
plt.xlabel("C (Regularization strength inverse)")
plt.ylabel("Accuracy")
plt.title("Effect of Regularization (C)")
plt.legend()
plt.show()

In [ ]:
best_C = C_values[np.argmax(list(test_acc.values()))]

l2_model = LogisticRegression(C=best_C, penalty='l2', solver='lbfgs', max_iter=1000)
l2_model.fit(x_train, y_train)

l2_pred = l2_model.predict(x_test)

print("L2 Accuracy:", accuracy_score(y_test, l2_pred))
print(classification_report(y_test, l2_pred))

In [ ]:
l1_model = LogisticRegression(C=best_C, penalty='l1', solver='saga', max_iter=1000)
l1_model.fit(x_train, y_train)

l1_pred = l1_model.predict(x_test)

print("L1 Accuracy:", accuracy_score(y_test, l1_pred))
print(classification_report(y_test, l1_pred))

In [ ]:
df_pre

In [ ]:
df

In [ ]:
X_num = df[['Age scaled', 'Future Potential scaled', 'Total_Stats Score']]
y = df['Performance_Tier_encoded']

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(x_train, y_train)

print("NB Accuracy:", nb.score(x_test, y_test) , '\n')

y_pred = nb.predict(x_test)
print("NB Accuracy Score:", accuracy_score(y_pred, y_test) , '\n')

print("NB report:", classification_report(y_test, y_pred) , '\n')

NB_cm = confusion_matrix(y_test, y_pred)
print(f'NB Confusion Matrix : \n {NB_cm}')